# RaceShift FFR Colab Training

Train and evaluate the RaceShift **Forward-Forward regressor** (no global backpropagation) on real Formula 1 laps.

How this notebook is organised:

1. Environment check and Google Drive mount. Code lives in `/content/RaceShift`; data, caches and artifacts persist in `Drive/RaceShiftData`.
2. Clone (or update) the project from GitHub and install it.
3. **Synthetic smoke test first.** It proves the pipeline before any race data is downloaded. Its numbers are never Formula 1 claims.
4. Collect a small real subset: 2022-2025 races at Bahrain, Silverstone and Monza.
5. **Baselines before the neural model.** Previous lap, rolling-five median, ridge and gradient boosting on the same leakage-safe table.
6. Train the production FFR ladder (512 → 384 → 256 → 192, 8/16/32/64 ordinal groups) with a chronological split: train ≤ 2023, validate 2024, test 2025.
7. Optional wider ablation and Hugging Face source inspection.
8. Copy the artifact back into the local app.

RaceShift FFR is NumPy-based. A GPU runtime is **optional**; a CPU or high-RAM runtime is fine. Colab does not guarantee runtime duration, so everything important is written to Drive as soon as it is produced.

In [ ]:
import sys, platform
print('Python:', sys.version.split()[0], '|', platform.platform())
try:
    import torch  # only informational: RaceShift FFR does not use torch for training
    print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
except Exception as exc:  # noqa: BLE001
    print('torch not available (fine for RaceShift FFR):', exc)

## 1. Mount Google Drive

Everything under `RaceShiftData` survives runtime resets: the FastF1 cache, raw parquet files, processed tables and trained artifacts.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/RaceShiftData')
RAW = DRIVE_ROOT / 'raw' / 'fastf1'
CACHE = DRIVE_ROOT / 'cache' / 'fastf1'
PROCESSED = DRIVE_ROOT / 'processed'
ARTIFACTS = DRIVE_ROOT / 'artifacts'
for p in (RAW, CACHE, PROCESSED, ARTIFACTS):
    p.mkdir(parents=True, exist_ok=True)
print('Persistent data root:', DRIVE_ROOT)

## 2. Get the RaceShift code

The project lives in GitHub. Change `REPO` / `BRANCH` if you are working from a fork or a feature branch. Re-running this cell pulls the latest commit.

In [ ]:
import os, subprocess
REPO = 'https://github.com/parthd25/raceshift'
BRANCH = 'main'
PROJECT = Path('/content/RaceShift')
if PROJECT.exists():
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO, str(PROJECT)], check=True)
os.chdir(PROJECT)
print(subprocess.run(['git', 'log', '--oneline', '-n', '1'], capture_output=True, text=True).stdout)

In [ ]:
%pip install -q -e '.[research]'
import raceshift
print('raceshift', raceshift.__version__)

## 3. Synthetic smoke test (no race data yet)

This verifies `import → features → chronological split → local Forward-Forward training → artifact → evaluation` in under a minute. The metrics describe a synthetic fixture and must never be quoted as Formula 1 performance.

In [ ]:
SMOKE = Path('/content/smoke')
SMOKE.mkdir(exist_ok=True)
!python scripts/make_synthetic_fixture.py --output {SMOKE}/synthetic_fixture.csv
!python scripts/train_baselines.py --input {SMOKE}/synthetic_fixture.csv --output {SMOKE}/baselines --train-end 2023 --val-year 2024 --test-year 2025
!python scripts/train_ffr.py --input {SMOKE}/synthetic_fixture.csv --config configs/ffr_demo.json --output {SMOKE}/ffr_demo --train-end 2023 --val-year 2024 --test-year 2025

## 4. Collect the first real Formula 1 subset

Three very different circuits across four seasons keep the first real run small enough to debug. FastF1 caches every session in Drive, so re-runs are fast. Expand `--years` / `--events` (or drop `--events` for full seasons) once this pipeline is clean.

Requires `--session R` race sessions; qualifying and practice can be added later as separate experiments.

In [ ]:
!python scripts/fetch_fastf1_seasons.py \
  --years 2022-2025 \
  --session R \
  --events Bahrain Silverstone Monza \
  --output "{RAW}" \
  --cache "{CACHE}"

In [ ]:
import glob
import pandas as pd
files = sorted(glob.glob(str(RAW / '*.parquet')))
print(len(files), 'session files')
laps = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
LAPS = PROCESSED / 'f1_laps_smoke.parquet'
laps.to_parquet(LAPS, index=False)
print(len(laps), 'raw laps ->', LAPS)
laps.groupby('season')['event'].nunique()

## 5. Baselines first

Do not judge the neural model until these exist. The baseline script uses the **same** full-context feature table, chronological split and train-only preprocessing as `train_ffr.py`, so the comparison is fair.

In [ ]:
!python scripts/train_baselines.py --input "{LAPS}" --output "{ARTIFACTS}/baselines_smoke" --train-end 2023 --val-year 2024 --test-year 2025

## 6. Train the production Forward-Forward ladder

`configs/ffr_production.json`: 512 → 384 → 256 → 192 hidden nodes, 8/16/32/64 ordinal goodness groups, explicit local Adam per layer, closed-form ridge readout, 80% validation-calibrated interval. No `Tensor.backward()` or `autograd.grad()` anywhere in the training path (a unit test enforces this).

The artifact folder is written directly to Drive.

In [ ]:
!python scripts/train_ffr.py \
  --input "{LAPS}" \
  --config configs/ffr_production.json \
  --output "{ARTIFACTS}/raceshift_ffr_smoke" \
  --train-end 2023 --val-year 2024 --test-year 2025

In [ ]:
import json
ffr = json.loads((ARTIFACTS / 'raceshift_ffr_smoke' / 'metrics.json').read_text())
base = json.loads((ARTIFACTS / 'baselines_smoke' / 'metrics.json').read_text())
rows = [{'model': 'RaceShift FFR', **{k: ffr['test'][k] for k in ('mae_s', 'rmse_s', 'p90_ae_s')}, 'coverage80': ffr['test']['interval80_coverage']}]
rows += [{'model': name, **{k: r['test'][k] for k in ('mae_s', 'rmse_s', 'p90_ae_s')}} for name, r in base['models'].items()]
pd.DataFrame(rows).set_index('model').sort_values('mae_s')

## 7. Optional: wider research ladder

Only run after the production configuration has been evaluated. `configs/ffr_colab_large.json` is 1024 → 768 → 512 → 384 → 256. Reduce node counts if the runtime runs out of RAM.

In [ ]:
# !python scripts/train_ffr.py --input "{LAPS}" --config configs/ffr_colab_large.json --output "{ARTIFACTS}/raceshift_ffr_large" --train-end 2023 --val-year 2024 --test-year 2025

## 8. Optional: inspect large Hugging Face sources before downloading

These datasets are multi-gigabyte. Listing is free; downloading is opt-in and stays outside git. Check each source's licence in `docs/DATA_AND_MODEL_FINDINGS.md` first.

In [ ]:
!python scripts/fetch_hf.py FlorindoDev/f1_corner_telemetry_2024_2025 --list
!python scripts/fetch_hf.py VforVitorio/f1-strategy-dataset --list
!python scripts/fetch_hf.py tobil/imsa --list

## 9. Use the trained artifact locally

Download the folder `RaceShiftData/artifacts/raceshift_ffr_smoke` (six files plus `test_predictions.csv`) and copy it into the project's `artifacts/` directory on your machine, for example `artifacts/raceshift_ffr_2025test/`. The local API lists every complete artifact under `GET /api/models`, and the Forecast page lets you pick it. Real artifacts are labelled by their `data_source`; the packaged `raceshift_ffr_demo` stays labelled synthetic.

Reporting rules: quote future-season test metrics with the holdout year named, always next to the baselines from step 5, and never quote step-3 synthetic numbers as Formula 1 results.